In [26]:
from pathlib import Path

from score.score import run
from score.data import load_metrics

from plot.metric_network_mean import plot_raw_network_metric
from plot.utils import load_ranks
from plot.rank_map import make_map
from plot.score_trends import make_trends
from plot.trend_fits import make_trend_fits
from plot.trend_map import make_trend_map
from plot.igc20_map import make_agreement_map
from plot.lat_ranks import (
    make_lat_ranks,
    make_lat_trends,
)
from plot.station_map import make_station_map
from config import load_config, Variant

import pandas as pd
IGS_CSV = "data/igs_stations.csv"

In [27]:
# Figure 1
# Load stations from IGS stations csv
stations = pd.read_csv(IGS_CSV)
stations["station"] = stations["Site Name"].astype(str).str[:4]
make_station_map(
    stations_file="data/stations.txt",
    igs_csv=IGS_CSV,
    plots_dir=None,
    include_titles=False
)

In [28]:

# PPP-augmented metrics
ppp_dir = Path("./results") / "ppp" / Variant.topsis.name
ppp_config = Path("scenarios/ppp.yaml")
ppp_config = load_config(ppp_config)

# Figure 3
raw_df = load_metrics(ppp_config)
plot_raw_network_metric(
    raw_df,
    "phase_wrms"
)

In [29]:

# Figure 4
ranks, metric_cols = load_ranks(ppp_config, Variant.topsis, stations)
make_agreement_map(
    ranks,
    metric_cols,
    ppp_dir,
    stations,
    title="PPP-augmented ranks in IGc20 clusters"
)

In [30]:

# Preprocessed metrics
preprocessed_dir = Path("./results") / "preprocessed" / Variant.topsis.name
preprocessed_config = Path("scenarios/preprocessed.yaml")
preprocessed_config = load_config(preprocessed_config)
preproc_ranks, preproc_metric_cols = load_ranks(preprocessed_config, Variant.topsis, stations)

# Figure 5
make_agreement_map(
    preproc_ranks,
    preproc_metric_cols,
    preprocessed_dir,
    stations,
    title="Preprocessed metrics - ranks in IGc20 clusters"
)

In [31]:
# Table 4
from IPython.display import HTML
df = ranks[["station", "score", "rank"]]
df = pd.concat([df.head(5), df.tail(5)])
HTML(df.to_html(index=False))


station,score,rank
BRUX,0.587274,1
LROC,0.582981,2
HERS,0.580271,3
STJ3,0.580197,4
VNDP,0.578982,5
DARW,0.381666,142
CIBG,0.371659,143
UTQI,0.357291,144
SALU,0.337021,145
POHN,0.324319,146


In [32]:
# Figure 6
make_map(
    ranks,
    metric_cols,
    title="Station Ranks - PPP-augmented",
)

In [33]:
# Figure 7
make_lat_ranks(
    ranks,
    title="PPP-augmented metrics - Average rank by geographic latitude"
)

In [34]:

# Run PPP for each day over the timeseries
ppp_windowed_dir = Path("./results") / "ppp_windowed" / Variant.topsis.name
ppp_windowed_config = Path("scenarios/ppp_windowed.yaml")
ppp_windowed_config = load_config(ppp_windowed_config)

run(ppp_windowed_config)

In [35]:
# Table 5
ppp_windowed_ranks, ppp_windowed_metric_cols = load_ranks(ppp_windowed_config, Variant.topsis, stations)
_, fits = make_trend_fits(ppp_windowed_ranks, ppp_windowed_dir, ppp_windowed_config.name, ppp_windowed_config.weights)

n=5
station_rank = ranks[["station", "rank"]].rename(columns={"rank": "rank"})
table = pd.concat([fits.head(n), fits.tail(n)])[["station", "slope_per_year"]]
table["direction"] = table["slope_per_year"].apply(lambda x: "Increasing" if x > 0 else "Decreasing")
table["slope_per_year"] = table["slope_per_year"].round(3)
table = table.merge(station_rank, on="station")[["rank", "station", "slope_per_year", "direction"]]
table

,rank,station,slope_per_year,direction
0,121,AIRA,0.022,Increasing
1,33,YEBE,0.011,Increasing
2,68,POLV,0.011,Increasing
3,27,PALM,0.010,Increasing
4,112,PTVL,0.009,Increasing
5,110,SCTB,-0.013,Decreasing
6,142,DARW,-0.014,Decreasing
7,77,CPVG,-0.015,Decreasing
8,84,LCK4,-0.017,Decreasing
9,59,ISPA,-0.018,Decreasing


In [36]:
# Figure 8
make_trend_map(
    fits,
    ranks,
    title=("Score trend per station")
)

In [37]:
# Figure 9
make_lat_trends(
    fits,
    ppp_windowed_ranks,
    title=("Mean score trend by geographic latitude (20 degree bins)"),
)

In [38]:
def compare_metrics(ranks, metric_cols, station_a, station_b):
    subset = ranks[ranks["station"].isin([station_a, station_b])][["station"] + metric_cols]
    pivot = subset.set_index("station")[metric_cols].T.reset_index().rename(columns={"index":
    "metric"})
    weights_df = pd.DataFrame(ppp_config.weights.items(), columns=["metric", "weight"])
    table = pivot.merge(weights_df, on="metric").sort_values("weight", ascending=False)
    
    return table

# Table 6
table = compare_metrics(ranks, metric_cols, "DARW", "TOW2")
table[["metric", "weight", "DARW", "TOW2"]].round(3)

,metric,weight,DARW,TOW2
0,phase_wrms,0.225,0.255,0.471
1,amb_resets,0.175,0.467,0.516
2,code_wrms,0.100,0.343,0.567
3,h_conv,0.100,0.370,0.510
4,v_conv,0.100,0.458,0.484
5,satellite_gaps,0.075,0.352,0.907
6,uptime,0.075,0.838,0.939
7,cn0,0.050,0.282,0.432
8,mp1,0.017,0.467,0.454
9,mp2,0.017,0.508,0.450


In [39]:
# Figure 10
make_trends(
    ppp_windowed_ranks,
    metric_cols,
    title=("Station score over time - DARW"),
    stations=["DARW"],
)

In [40]:
# Table 7
table = compare_metrics(ranks, metric_cols, "RIO2", "FALK")
table[["metric", "weight", "RIO2", "FALK"]].round(3)

,metric,weight,RIO2,FALK
0,phase_wrms,0.225,0.586,0.475
1,amb_resets,0.175,0.521,0.510
2,code_wrms,0.100,0.626,0.432
3,h_conv,0.100,0.490,0.499
4,v_conv,0.100,0.503,0.490
5,satellite_gaps,0.075,0.574,0.317
6,uptime,0.075,0.969,0.993
7,cn0,0.050,0.489,0.220
8,mp1,0.017,0.539,0.462
9,mp2,0.017,0.494,0.441
